<a href="https://colab.research.google.com/github/chaseshallow/Ansci4040FinalClassProject/blob/main/RunPipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Run the whole pipeline: one or many videos

Runs **steps 01–05 for every video below, then step 06** (pool), with one call: the same functions and the same output folders as the step notebooks, so the results are identical. Use the step notebooks to check a new setting by eye; use this one to process videos (other cows) once the settings are right.

- `Test = True` runs the "a" versions (4 minutes from `TestStart`; folders `Step02aDetectTest` … `PoolTest`); `False` the full clip (`Step02bDetectFull` … `Pool`).
- Every step **overwrites** its folder for that video. Only step 03 continues a run that stopped part-way: if Colab stops, set `FromStep=3` for that video (so step 02 is not redone) and run cells 1–3 again.
- A video that fails is reported at the end; the others still run.

A full hour takes **~1.5–2 h on a T4** (mostly step 03). Needs a **T4 GPU** and `HF_TOKEN` (DINOv3). The folder is shared: one person runs a step at a time. Edit cell 2 and rerun; don't add cells.

In [ ]:
# 1. Setup: mount Drive, get the pipeline code from GitHub, install the model libraries.
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/chaseshallow/Ansci4040FinalClassProject.git"
RepoBranch = "main"
RepoDir = "/content/Ansci4040FinalClassProject"

if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
%pip install -q -r "{RepoDir}/CalvingPen/requirements.txt"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
from CalvingPipeline.Pipeline import RunVideos
from CalvingPipeline.Paths import ListVideos, OutputRoot
from IPython.display import display
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

try:
    HfToken = userdata.get("HF_TOKEN")   # from Colab Secrets; never paste the token here
except Exception:
    HfToken = None
print("HF_TOKEN:", "found" if HfToken else "not set - DINOv2 fallback will be used")

In [ ]:
# 2. Settings: the only cell to edit. One line per video (one calving, one cow).
display(ListVideos())   # the calving videos on the Shared Drive and whether they already have outputs

Videos = [
    # The reference video: its clip is already cut, so start at step 02. Its output folder keeps the old name.
    dict(VideoStem="2026-09-26 00-00-00 Calving Pen 2", FromStep=2),
    # A new video: VideoFile = file name from the table above; ClipStart / ClipEnd = when the calving
    # happens (HH:MM:SS into the video; find it with Step01PrepareClip's timeline sheet). Output folder =
    # the file name without .avi.
    # dict(VideoFile="2026-09-26-CalvingPen3(1).avi", ClipStart="01:20:00", ClipEnd="02:40:00"),
]
Test = False          # True = 4-minute test from TestStart (the "a" folders); False = the whole clip
TestStart = 1080      # seconds into the clip, only for Test = True
Pool = True           # run step 06 over all video folders at the end
Preview = True        # step 03 preview video (a little slower)

In [ ]:
# 3. Run: all steps for every video, then the pool.
import pandas as pd
from IPython.display import Image, display

Results = RunVideos(Videos, Test=Test, Pool=Pool, HfToken=HfToken, TestStart=TestStart, Preview=Preview)

if Pool:
    PoolDir = OutputRoot / ("PoolTest" if Test else "Pool")
    display(pd.read_csv(PoolDir / "PoolSummary.csv"))
    if (PoolDir / "SelfCheck.png").exists():
        display(Image(str(PoolDir / "SelfCheck.png")))

## About this step

**Functions** (`CalvingPen/CalvingPipeline/Pipeline.py`): `RunVideo(VideoStem or VideoFile, …)` runs steps 01–05 for one video; `RunVideos(Videos, …)` runs it for a list and then `BuildPool` (step 06). Each step is also its own function and can be called alone: `PrepareClips`, `DetectCows`, `TrackCows`, `CropCows`, `EmbedCrops`, `BuildPool`.

**Settings** are the defaults of the step notebooks (2 fps, SAM chunks of 2 min, ≥ 1 crop per 15 s, DINOv3). To change one for all videos, pass it on, e.g. `RunVideos(Videos, TargetFps=1, ...)`.

**Per video, check afterwards:** `Step03bTrackFull/TrackSummary.csv` (few long tracks, low `YoloDisagrees`) and `Step04bCropFull/Sheets/Overview.jpg` (only cows, no calf). After editing a video's `Identities.csv`, rebuild only the pool with `Videos = []` and `Pool = True`.

Full write-up, glossary and all models: `CalvingPen/README.md` in the repo.